# Kafka broker transient experiment plots

This notebook plots the CSV results produced by `com.myexam.app.qesm.App`. It covers:

- baseline queue behaviour;
- the effect of `BatchSize` on gateway and service backlog;
- the effect of `Timeout` on gateway and service backlog; and
- cumulative batching-induced idle behaviour.

Run the Java experiments from the project root before running this notebook.

## 1. Generate the outcome CSV files

From the project root:

```bash
mvn org.codehaus.mojo:exec-maven-plugin:3.5.0:java \
  -Dexec.mainClass=com.myexam.app.qesm.App
```

The command creates or replaces the required files in `outcomes/`.

Install the notebook dependencies and launch Jupyter with:

```bash
python3 -m pip install jupyter pandas matplotlib
jupyter notebook draw_plots.ipynb
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (11, 6),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.title_fontsize": 10,
    "legend.fontsize": 9,
})

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "pom.xml").exists():
            return candidate
    raise FileNotFoundError("Could not find the project folder containing pom.xml")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTCOMES_DIR = PROJECT_ROOT / "outcomes"

OUTCOME_FILES = {
    "baseline": "baseline_queue_behaviour.csv",
    "batch_size_10": "batch_size_10_backlog.csv",
    "batch_size_30": "batch_size_30_backlog.csv",
    "timeout_12_5": "timeout_12_5_backlog.csv",
    "timeout_35": "timeout_35_backlog.csv",
    "batching_idle": "batching_induced_idle.csv",
}

def finish_plot(fig):
    fig.tight_layout()
    plt.show()

## 2. Load and validate the results

In [ ]:
missing_files = [
    OUTCOMES_DIR / filename
    for filename in OUTCOME_FILES.values()
    if not (OUTCOMES_DIR / filename).is_file()
]
if missing_files:
    missing = "\n".join(f"- {path}" for path in missing_files)
    raise FileNotFoundError(
        "Missing experiment results. Run com.myexam.app.qesm.App first:\n" + missing
    )

results = {
    name: pd.read_csv(OUTCOMES_DIR / filename)
    for name, filename in OUTCOME_FILES.items()
}

normal_columns = {"time", "meanMsgsAtGateway", "meanAtService"}
idle_columns = {
    "time",
    "cumulativeEmpty",
    "cumulativeBatchingIdleReward",
    "batchingIdleTime",
    "batchingIdleFraction",
}

for name, frame in results.items():
    required = idle_columns if name == "batching_idle" else normal_columns
    missing_columns = required.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{OUTCOME_FILES[name]} is missing columns: {sorted(missing_columns)}")
    if frame.empty:
        raise ValueError(f"{OUTCOME_FILES[name]} contains no data rows")
    if not frame["time"].is_monotonic_increasing:
        raise ValueError(f"Time is not ordered in {OUTCOME_FILES[name]}")

summary = pd.DataFrame([
    {
        "experiment": name,
        "rows": len(frame),
        "start_time": frame["time"].iloc[0],
        "end_time": frame["time"].iloc[-1],
    }
    for name, frame in results.items()
])
display(summary)

## 3. Baseline queue behaviour

Baseline parameters: `BatchSize=20`, `Timeout=25`, `Overhead=2`, and `Stability=30`. The two normal transient rewards are shown in separate figures.

In [ ]:
baseline = results["baseline"]

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(baseline["time"], baseline["meanMsgsAtGateway"], color="tab:blue", linewidth=2)
ax.set_title("Baseline: messages waiting at the gateway")
ax.set_xlabel("Simulation time")
ax.set_ylabel("Mean MsgsAtGateway")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
finish_plot(fig)

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(baseline["time"], baseline["meanAtService"], color="tab:orange", linewidth=2)
ax.set_title("Baseline: messages at service")
ax.set_xlabel("Simulation time")
ax.set_ylabel("Mean AtService")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
finish_plot(fig)

## 4. BatchSize effect on backlog

This comparison changes only `BatchSize`. The baseline result supplies the `BatchSize=20` series. Gateway backlog and service backlog are plotted separately.

In [ ]:
batch_size_results = {
    10: results["batch_size_10"],
    20: results["baseline"],
    30: results["batch_size_30"],
}

fig, ax = plt.subplots(figsize=(11, 6))
for batch_size, frame in batch_size_results.items():
    ax.plot(frame["time"], frame["meanMsgsAtGateway"], label=f"{batch_size}")
ax.set_title("BatchSize effect on gateway backlog (Timeout=25)")
ax.set_xlabel("Simulation time")
ax.set_ylabel("Mean MsgsAtGateway")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.legend(title="Batch size")
finish_plot(fig)

fig, ax = plt.subplots(figsize=(11, 6))
for batch_size, frame in batch_size_results.items():
    ax.plot(frame["time"], frame["meanAtService"], label=f"{batch_size}")
ax.set_title("BatchSize effect on service backlog (Timeout=25)")
ax.set_xlabel("Simulation time")
ax.set_ylabel("Mean AtService")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.legend(title="Batch size")
finish_plot(fig)

## 5. Timeout effect on backlog

This comparison changes only `Timeout`. The baseline result supplies the `Timeout=25` series. Gateway backlog and service backlog are plotted separately.

In [ ]:
timeout_results = {
    12.5: results["timeout_12_5"],
    25.0: results["baseline"],
    35.0: results["timeout_35"],
}

fig, ax = plt.subplots(figsize=(11, 6))
for timeout, frame in timeout_results.items():
    label = f"{timeout:g}"
    ax.plot(frame["time"], frame["meanMsgsAtGateway"], label=label)
ax.set_title("Timeout effect on gateway backlog (BatchSize=20)")
ax.set_xlabel("Simulation time")
ax.set_ylabel("Mean MsgsAtGateway")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.legend(title="Timeout")
finish_plot(fig)

fig, ax = plt.subplots(figsize=(11, 6))
for timeout, frame in timeout_results.items():
    label = f"{timeout:g}"
    ax.plot(frame["time"], frame["meanAtService"], label=label)
ax.set_title("Timeout effect on service backlog (BatchSize=20)")
ax.set_xlabel("Simulation time")
ax.set_ylabel("Mean AtService")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
ax.legend(title="Timeout")
finish_plot(fig)

## 6. Batching-induced idle study

This study uses the cumulative watcher rewards `Empty;If(Idle==1&&NotEmpty==1,2,0)`. The Java analysis divides the second reward by two to obtain `batchingIdleTime`. Each cumulative or derived quantity is plotted separately because the values have different scales and interpretations.

In [ ]:
idle = results["batching_idle"]

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(idle["time"], idle["cumulativeEmpty"], color="tab:blue", linewidth=2)
ax.set_title("Cumulative empty-service reward")
ax.set_xlabel("Simulation time")
ax.set_ylabel("Cumulative Empty")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
finish_plot(fig)

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(
    idle["time"],
    idle["cumulativeBatchingIdleReward"],
    color="tab:orange",
    linewidth=2,
)
ax.set_title("Raw cumulative batching-idle watcher reward")
ax.set_xlabel("Simulation time")
ax.set_ylabel("Cumulative watcher reward")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
finish_plot(fig)

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(idle["time"], idle["batchingIdleTime"], color="tab:green", linewidth=2)
ax.set_title("Cumulative batching-induced idle time")
ax.set_xlabel("Simulation time")
ax.set_ylabel("Idle time")
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
finish_plot(fig)

fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(
    idle["time"],
    idle["batchingIdleFraction"],
    color="tab:red",
    linewidth=2,
)
ax.set_title("Fraction of elapsed time spent idle while messages wait")
ax.set_xlabel("Simulation time")
ax.set_ylabel("Idle-time fraction")
ax.set_xlim(left=0)
ax.set_ylim(0, 1)
finish_plot(fig)